In [ ]:
# Pipeline parameters — overridden by job base_parameters when run via DAB.
dbutils.widgets.text("catalog", "actuarial")
dbutils.widgets.text("schema", "dev")
dbutils.widgets.text("volume_name", "raw_files")
dbutils.widgets.text("bronze_write_mode", "overwrite")
dbutils.widgets.text("overwrite_schema", "true")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume_name = dbutils.widgets.get("volume_name")
bronze_write_mode = dbutils.widgets.get("bronze_write_mode")
overwrite_schema = dbutils.widgets.get("overwrite_schema").lower() == "true"
volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"

print(f"catalog={catalog}  schema={schema}  volume_path={volume_path}")
print(f"bronze_write_mode={bronze_write_mode}  overwrite_schema={overwrite_schema}")

# Data Quality — Bronze Layer Checks

## Step 4 - Identify Duplicate Records Across Bronze Tables


In [ ]:
from pyspark.sql import functions as F

# Exclude audit columns added during ingestion — not part of business identity
AUDIT_COLS = {"ingestion_timestamp", "source_file_name"}

# ── 1. Discover all bronze tables ────────────────────────────────────────────────
bronze_tables = sorted([
    row.tableName
    for row in spark.sql(f"SHOW TABLES IN {catalog}.{schema}").collect()
    if row.tableName.startswith("bronze_")
])

print(f"Scanning {len(bronze_tables)} bronze table(s) for duplicates …\n")

summary     = []
total_dirty = 0

# ── 2. Duplicate scan ──────────────────────────────────────────────────────────
for table_name in bronze_tables:
    full_name  = f"{catalog}.{schema}.{table_name}"
    df         = spark.table(full_name)

    # Derive business columns before any action (avoids extra Analyze RPC)
    biz_cols   = [c for c in df.schema.fieldNames() if c not in AUDIT_COLS]
    total_rows = df.count()

    # Groups of identical business-column rows that appear more than once
    dupes_df = (
        df.select(biz_cols)
          .groupBy(biz_cols)
          .agg(F.count("*").alias("duplicate_count"))
          .filter(F.col("duplicate_count") > 1)
          .orderBy(F.col("duplicate_count").desc())
    )

    # Single action — count duplicate groups AND total affected rows together
    agg = dupes_df.agg(
        F.count("*").alias("dupe_groups"),
        F.coalesce(F.sum("duplicate_count"), F.lit(0)).alias("dupe_rows")
    ).collect()[0]

    dupe_groups  = agg["dupe_groups"]
    dupe_rows    = agg["dupe_rows"]
    rows_to_drop = dupe_rows - dupe_groups   # extra copies that need removing

    status = "✅ Clean" if dupe_groups == 0 else f"⚠️  Duplicates found"
    total_dirty += (1 if dupe_groups > 0 else 0)

    summary.append({
        "table":        full_name,
        "total_rows":   total_rows,
        "dupe_groups":  dupe_groups,
        "rows_to_drop": rows_to_drop,
        "dupes_df":     dupes_df if dupe_groups > 0 else None
    })

    print(f"  {full_name}")
    print(f"    Total rows         : {total_rows:,}")
    print(f"    Duplicate groups   : {dupe_groups:,}")
    print(f"    Extra rows to drop : {rows_to_drop:,}")
    print(f"    Status             : {status}")
    print()

# ── 3. Overall summary ──────────────────────────────────────────────────────────
print(f"{'─'*60}")
if total_dirty == 0:
    print("✅ All bronze tables are clean — no duplicates found.")
else:
    print(f"⚠️  {total_dirty} table(s) contain duplicate records.")
    print("   Run the deduplication cell to clean them.")
print(f"{'─'*60}")

# ── 4. Show sample duplicate rows for dirty tables ───────────────────────────
for s in summary:
    if s["dupes_df"] is not None:
        print(f"\nSample duplicates — {s['table']}  (top by count):")
        display(s["dupes_df"].limit(20))

## Comprehensive Data Quality Report


In [ ]:
from pyspark.sql import functions as F

AUDIT_COLS = {"ingestion_timestamp", "source_file_name"}

# ── Load all tables once ───────────────────────────────────────────────────────────
claims    = spark.table(f"{catalog}.{schema}.bronze_claims_bordereau")
events    = spark.table(f"{catalog}.{schema}.bronze_cyclone_events")
premiums  = spark.table(f"{catalog}.{schema}.bronze_premium_bordereau")
risk_zone = spark.table(f"{catalog}.{schema}.bronze_risk_zone_lookup")

# Capture schemas ONCE before any actions — avoids repeated Analyze RPCs
claims_cols    = [c for c in claims.schema.fieldNames()    if c not in AUDIT_COLS]
events_cols    = [c for c in events.schema.fieldNames()    if c not in AUDIT_COLS]
premium_cols   = [c for c in premiums.schema.fieldNames()  if c not in AUDIT_COLS]
riskzone_cols  = [c for c in risk_zone.schema.fieldNames() if c not in AUDIT_COLS]

issues = []

print("═" * 70)
print("  DATA QUALITY REPORT — Bronze Layer")
print("═" * 70)

# ── 1. NULL / MISSING VALUE ANALYSIS ────────────────────────────────────────────────
print("\n📋 1. NULL / MISSING VALUE ANALYSIS")
print("─" * 70)

for tbl_name, df, biz_cols in [
    ("bronze_claims_bordereau",  claims,    claims_cols),
    ("bronze_cyclone_events",    events,    events_cols),
    ("bronze_premium_bordereau", premiums,  premium_cols),
    ("bronze_risk_zone_lookup",  risk_zone, riskzone_cols),
]:
    total    = df.count()
    null_row = df.agg(*[
        F.sum(F.col(c).isNull().cast("int")).alias(c) for c in biz_cols
    ]).collect()[0].asDict()

    dirty = {c: n for c, n in null_row.items() if n > 0}
    status = f"⚠️  {len(dirty)} column(s) have NULLs" if dirty else "✅ No NULLs"
    print(f"  {tbl_name:42s} {status}")
    for col_name, n in dirty.items():
        pct = round(n / total * 100, 1)
        sev = "HIGH" if pct > 10 else "MEDIUM"
        print(f"    └── {col_name}: {n:,} nulls ({pct}%)")
        issues.append({"table": tbl_name, "check_type": "NULL values",
                       "column": col_name, "severity": sev,
                       "details": f"{n:,} nulls ({pct}% of {total:,} rows)"})

# ── 2. BUSINESS RULE VIOLATIONS ──────────────────────────────────────────────────────
print("\n📋 2. BUSINESS RULE VIOLATIONS")
print("─" * 70)

def rule(label, n, table, col, msg):
    """Log a business rule check and append to issues list if violated."""
    status = "✅ OK" if n == 0 else f"⚠️  {n:,} rows"
    print(f"  {label:55s} {status}")
    if n > 0:
        issues.append({"table": table, "check_type": "Business rule",
                       "column": col, "severity": "HIGH", "details": msg(n)})

def parse_date(col_name):
    """Parse string dates in 'MM/dd/yyyy' or 'yyyy-MM-dd' format gracefully."""
    return F.coalesce(
        F.expr(f"try_to_date(`{col_name}`, 'MM/dd/yyyy')"),
        F.expr(f"try_to_date(`{col_name}`, 'yyyy-MM-dd')")
    )

def try_num(col_name):
    """Safe cast to DOUBLE — returns NULL for 'N/A', blanks, or any non-numeric string."""
    return F.expr(f"try_cast(`{col_name}` AS DOUBLE)")

rule("claims: date_of_loss > reported_date",
     claims.filter(parse_date("date_of_loss") > parse_date("reported_date")).count(),
     "bronze_claims_bordereau", "date_of_loss / reported_date",
     lambda n: f"{n:,} claims where loss date is after reported date")

rule("claims: incurred_amount < 0",
     claims.filter(try_num("incurred_amount") < 0).count(),
     "bronze_claims_bordereau", "incurred_amount",
     lambda n: f"{n:,} claims with negative incurred amount")

rule("claims: paid_to_date > incurred_amount",
     claims.filter(try_num("paid_to_date") > try_num("incurred_amount")).count(),
     "bronze_claims_bordereau", "paid_to_date",
     lambda n: f"{n:,} claims where amount paid exceeds amount incurred")

rule("premiums: policy_start_date >= policy_end_date",
     premiums.filter(parse_date("policy_start_date") >= parse_date("policy_end_date")).count(),
     "bronze_premium_bordereau", "policy_start_date / policy_end_date",
     lambda n: f"{n:,} policies with start date on or after end date")

rule("premiums: sum_insured <= 0",
     premiums.filter(try_num("sum_insured") <= 0).count(),
     "bronze_premium_bordereau", "sum_insured",
     lambda n: f"{n:,} policies with zero or negative sum insured")

rule("premiums: annual_premium <= 0",
     premiums.filter(try_num("annual_premium") <= 0).count(),
     "bronze_premium_bordereau", "annual_premium",
     lambda n: f"{n:,} policies with zero or negative annual premium")

rule("cyclone_events: start_date > end_date",
     events.filter(parse_date("start_date") > F.col("end_date")).count(),
     "bronze_cyclone_events", "start_date / end_date",
     lambda n: f"{n:,} events where start date is after end date")

# ── 3. REFERENTIAL INTEGRITY ───────────────────────────────────────────────────────────
print("\n📋 3. REFERENTIAL INTEGRITY")
print("─" * 70)

rule("claims.policy_id → premium_bordereau (policy_id)",
     claims.join(premiums.select("policy_id").distinct(), "policy_id", "left_anti").count(),
     "bronze_claims_bordereau", "policy_id",
     lambda n: f"{n:,} claims reference a policy_id missing from premium_bordereau")

rule("claims.event_id → cyclone_events (non-null only)",
     claims.filter(F.col("event_id").isNotNull())
           .join(events.select("event_id").distinct(), "event_id", "left_anti").count(),
     "bronze_claims_bordereau", "event_id",
     lambda n: f"{n:,} claims reference an event_id missing from cyclone_events")

rule("premium.postcode → risk_zone_lookup (postcode)",
     premiums.withColumn("postcode", F.col("postcode").cast("int")).join(risk_zone.select("postcode").distinct(), "postcode", "left_anti").count(),
     "bronze_premium_bordereau", "postcode",
     lambda n: f"{n:,} premiums reference a postcode missing from risk_zone_lookup")

# ── 4. ISSUES SUMMARY TABLE ───────────────────────────────────────────────────────────
print(f"\n{'═' * 70}")
print(f"  TOTAL ISSUES FOUND: {len(issues)}")
print(f"{'═' * 70}")

if issues:
    issues_df = spark.createDataFrame(issues).orderBy("severity", "table", "check_type")
    display(issues_df)
else:
    print("  ✅ All checks passed — no data quality issues detected.")

## Risk Zone Lookup — One Row per Postcode Check


In [ ]:
from pyspark.sql import functions as F

risk_zone = spark.table(f"{catalog}.{schema}.bronze_risk_zone_lookup")

total_rows      = risk_zone.count()
unique_postcodes = risk_zone.select("postcode").distinct().count()

print(f"Total rows        : {total_rows:,}")
print(f"Unique postcodes  : {unique_postcodes:,}")

if total_rows == unique_postcodes:
    print("\n✅ PASS — Exactly one row per postcode. Lookup table is clean.")
else:
    extra = total_rows - unique_postcodes
    print(f"\n⚠️  FAIL — {extra:,} extra row(s) detected ({total_rows} rows vs {unique_postcodes} unique postcodes).")
    print("\nPostcodes with more than one row:")
    duplicated_postcodes = (
        risk_zone
        .groupBy("postcode", "region_name", "wind_risk_band")
        .agg(F.count("*").alias("row_count"))
        .filter(F.col("row_count") > 1)
        .orderBy(F.col("row_count").desc())
    )
    display(duplicated_postcodes)

    # Show all conflicting rows (different values for the same postcode)
    print("\nConflicting values for the same postcode (different region / risk band):")
    postcode_counts = (
        risk_zone
        .groupBy("postcode")
        .agg(
            F.count("*").alias("row_count"),
            F.countDistinct("region_name").alias("distinct_regions"),
            F.countDistinct("wind_risk_band").alias("distinct_risk_bands")
        )
        .filter(F.col("row_count") > 1)
    )
    display(
        risk_zone
        .join(postcode_counts.select("postcode"), on="postcode", how="inner")
        .orderBy("postcode")
    )